# Ablation — 카탈로그 유형 플래그 제거실험

**방식**: [feature_catalog.csv](feature_catalog.csv) 의 **유형 플래그를 하나씩 꺼서**
같은 아키텍처(LGBM + swap 증강/TTA)·같은 fold·같은 ES(valid) 로 재학습한다 —
단일 변수는 피쳐셋뿐. 부스터는 LGBM 하나만 쓴다(대표·최속 — 3 부스터 대조는 Train 몫).

**판정** (recipe §7): holdout 5,748 에서 full 대비 per-row loss 의 paired bootstrap
95% CI (2000 resample), 채택 문턱 0.0005. holdout 은 stage2 보다 작아 **SE ≈ 0.0058,
CI 폭 1.37배** — 경계 판정은 보수적으로 읽을 것.

| arm | 답하는 질문 |
|---|---|
| full (Train 캐시 재사용) | 기준 |
| − 각 유형 (12개) | 그 유형이 실제로 벌고 있는가. `− 임베딩` = MiniLM 없이, `− TF-IDF 잠재` 등은 아티팩트 의존 절감 판단 |
| (선택) FLAGS_CUSTOM | 원하는 조합을 직접 정의 |

**읽기**: `− 유형` 의 Δ가 **+유의**면 그 유형이 벌고 있다(빼면 나빠짐). **노이즈**면 제거 후보.

**선행**: Features + Train 완료. **예상 시간**: 12 arm × ~2분 ≈ **~25분**. `QUICK=True` 는 배선 점검용.

In [ ]:
# ── 설정 + 로드 (Train 과 동일 경로) ──
import os, sys, json, time
import numpy as np
import pandas as pd
from tqdm import tqdm

SEED = 42
META_COLS = ['id', 'role', 'label', 'is_tie']
QUICK = False                       # True: n_estimators 400 배선 점검 (결론용 아님)
# ── 데이터 로드 — origin/ 3분할. 훈련=train, ES=valid, 평가=holdout (npz 미사용) ──
def labels_of(d):
    yy = np.select([d['winner_model_a'] == 1, d['winner_model_b'] == 1,
                    d['winner_tie'] == 1], [0, 1, 2], default=-1).astype(np.int64)
    assert (yy != -1).all(), 'invalid labels!'
    return yy

parts = {}
for name, n_expect in [('train', 46729), ('valid', 5000), ('holdout', 5748)]:
    d = pd.read_csv(os.path.join('origin', f'{name}.csv'))
    assert len(d) == n_expect, f'{name}.csv {len(d)}행 != {n_expect}'
    d['role'] = name
    parts[name] = d
df = pd.concat([parts['train'], parts['valid'], parts['holdout']], ignore_index=True)
assert df['id'].is_unique, 'id 중복!'
y = labels_of(df)
role = df['role'].to_numpy()
m_tr, m_va, m_ho = role == 'train', role == 'valid', role == 'holdout'
y_tr, y_va, y_ho = y[m_tr], y[m_va], y[m_ho]
print(f'train {m_tr.sum():,} / valid {m_va.sum():,} / holdout {m_ho.sum():,}')
print('라벨 분포(a/b/tie):', np.round(np.bincount(y, minlength=3) / len(y), 4))

feat = pd.read_parquet('features_57k.parquet')
assert (feat['id'].to_numpy() == df['id'].to_numpy()).all(), 'features_57k 정렬 불일치!'
F = feat.drop(columns=META_COLS)
catalog = pd.read_csv('feature_catalog.csv').fillna({'제외사유': ''})
assert set(catalog['feature']) == set(F.columns)
folds_df = pd.read_csv('arith_folds.csv')
assert (folds_df['id'].to_numpy() == df.loc[m_tr, 'id'].to_numpy()).all()
folds = folds_df['fold'].to_numpy()
sw_series = catalog.set_index('feature')['swap']
TYPES = list(catalog['유형'].unique())
BASE_USE = catalog['사용'] == 1
print('유형:', TYPES)

# full arm 캐시 — Train 산출물과 컬럼이 같으면 재학습 생략
FULL_CACHED, P0 = False, None
try:
    meta = json.load(open('model/meta.json', encoding='utf-8'))
    P0 = np.load('outputs/arith_experts.npz', allow_pickle=False)
    full_cols = catalog.loc[BASE_USE, 'feature'].tolist()
    FULL_CACHED = (meta['cols'] == full_cols and all(meta['flags'].values())
                   and bool((P0['ids_hold'] == df.loc[m_ho, 'id'].to_numpy()).all())
                   and not QUICK)
except Exception as e:
    print('Train 캐시 없음:', e)
print('full arm:', 'Train 캐시(lgbm) 재사용' if FULL_CACHED else '재학습')

In [ ]:
# ── swap 매핑 — 카탈로그의 swap 컬럼이 진실의 원천 (A↔B 교환 시 피쳐 거동) ──
def build_swap(cols, swap_series):
    neg = [c for c in cols if swap_series[c] == '부호반전']
    recip = [c for c in cols if swap_series[c] == '역수']
    pairs = []
    for c in cols:
        v = swap_series[c]
        if isinstance(v, str) and v.startswith('교환:') and c < v.split(':', 1)[1]:
            partner = v.split(':', 1)[1]
            assert partner in set(cols), f'교환 파트너 {partner} 미포함 — 유형 플래그가 쌍을 갈랐다'
            pairs.append((c, partner))
    inv = [c for c in cols if swap_series[c] == '불변']
    assert len(neg) + len(recip) + len(inv) + 2 * len(pairs) == len(cols), 'swap 분류 누락!'
    idx = {c: i for i, c in enumerate(cols)}
    neg_i = np.array([idx[c] for c in neg], int)
    rec_i = np.array([idx[c] for c in recip], int)
    a_i = np.array([idx[a] for a, _ in pairs], int)
    b_i = np.array([idx[b] for _, b in pairs], int)

    def swap_X(X):
        Xs = X.copy()
        if len(neg_i):
            Xs[:, neg_i] *= -1.0
        if len(rec_i):
            Xs[:, rec_i] = 1.0 / np.clip(Xs[:, rec_i], 1e-12, None)
        if len(a_i):
            Xs[:, a_i], Xs[:, b_i] = Xs[:, b_i].copy(), Xs[:, a_i].copy()
        return Xs
    info = {'pairs': pairs, 'neg': neg, 'recip': recip, 'inv': inv}
    return swap_X, info

def swap_p3(p):
    return p[:, [1, 0, 2]]

def swap_label(yy):
    return np.where(yy == 0, 1, np.where(yy == 1, 0, 2)).astype(yy.dtype)

# ── 평가 유틸 — §0 축 분해 항등식 + §7 paired bootstrap ──
EPS = 1e-15

def per_row_ll(p, yy):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    return -np.log(p[np.arange(len(yy)), yy])

def axis_report(p, yy):
    # 총 ll = tie축 + 승부축 (임의의 3-class 확률벡터에 성립하는 항등식, recipe §0)
    from sklearn.metrics import roc_auc_score
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    yy = np.asarray(yy)
    total = per_row_ll(p, yy).mean()
    t = np.clip(p[:, 2], EPS, 1 - EPS)
    it = yy == 2
    tie_axis = float(-np.where(it, np.log(t), np.log(1 - t)).mean())
    nz = ~it
    rA = np.clip(p[nz, 0] / (p[nz, 0] + p[nz, 1]), EPS, 1 - EPS)
    win_axis = float(-np.where(yy[nz] == 0, np.log(rA), np.log(1 - rA)).sum() / len(yy))
    assert abs(total - tie_axis - win_axis) < 1e-6, '축 분해 검산 실패'
    return {'총 ll': float(total), 'tie축': tie_axis, '승부축': win_axis,
            'tie AUC': float(roc_auc_score(it.astype(int), t)),
            '승부 AUC': float(roc_auc_score((yy[nz] == 0).astype(int), rA)),
            'acc%': float((p.argmax(1) == yy).mean() * 100),
            'tie recall%': float((p.argmax(1) == 2)[it].mean() * 100)}

def paired_bootstrap(ll_a, ll_b, n_boot=2000, seed=SEED):
    # Δ = mean(ll_a − ll_b) 의 95% CI. 음수면 a 가 좋다. CI 가 0 을 포함하면 노이즈.
    d = np.asarray(ll_a, np.float64) - np.asarray(ll_b, np.float64)
    rng = np.random.default_rng(seed)
    n = len(d)
    means = np.empty(n_boot)
    for i in range(n_boot):
        means[i] = d[rng.integers(0, n, n)].mean()
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def fmt_ci(mean, lo, hi, thresh=0.0005):
    sig = '유의' if (lo > 0 or hi < 0) else '노이즈'
    star = ' ★' if sig == '유의' and abs(mean) >= thresh else ''
    return f'{mean:+.4f} [{lo:+.4f},{hi:+.4f}] {sig}{star}'

import inspect
import lightgbm as lgb
N_LGB = 400 if QUICK else 3000
_LGB_NEW = 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters

def train_arm(cols):
    # Train.ipynb 의 LGBM 경로와 동일 — swap 증강 + ES valid + holdout TTA 예측
    swap_X, _ = build_swap(cols, sw_series)
    Xt = F.loc[m_tr, cols].to_numpy(np.float32)
    Xv = F.loc[m_va, cols].to_numpy(np.float32)
    Xh = F.loc[m_ho, cols].to_numpy(np.float32)
    p_hold = np.zeros((len(Xh), 3), np.float64)
    for k in np.sort(np.unique(folds)):
        tr = folds != k
        Xf = np.vstack([Xt[tr], swap_X(Xt[tr])])
        yf = np.hstack([y_tr[tr], swap_label(y_tr[tr])])
        m = lgb.LGBMClassifier(objective='multiclass', num_class=3, learning_rate=0.03,
                               n_estimators=N_LGB, num_leaves=63, min_child_samples=100,
                               colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                               reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)
        ekw = {'eval_X': Xv, 'eval_y': y_va} if _LGB_NEW else {'eval_set': [(Xv, y_va)]}
        m.fit(Xf, yf, **ekw, callbacks=[lgb.early_stopping(100, verbose=False)])
        p_hold += (m.predict_proba(Xh) + swap_p3(m.predict_proba(swap_X(Xh)))) / 2 / 5
    return p_hold.astype(np.float32)
print('정의 완료')

In [ ]:
# ── arm 구성 — full + 유형별 off (+선택 커스텀) ──
ARMS = {'full (전 유형 on)': {g: True for g in TYPES}}
for g in TYPES:
    ARMS[f'− {g}'] = {t: (t != g) for t in TYPES}

# 원하는 조합이 있으면 여기에 추가 (예: 무상태만 남기기)
FLAGS_CUSTOM = {}
# FLAGS_CUSTOM['무상태만'] = {t: (t not in ('임베딩', 'TF-IDF 잠재')) for t in TYPES}  # 예시
ARMS.update(FLAGS_CUSTOM)

def cols_of(flags):
    m = catalog['유형'].map(flags) & BASE_USE
    return catalog.loc[m, 'feature'].tolist()

for k, fl in ARMS.items():
    print(f'  {k:<24} {len(cols_of(fl)):>3}종')

In [ ]:
# ── 실행 ──
results = {}
t_all = time.time()
for name, flags in ARMS.items():
    t0 = time.time()
    cols = cols_of(flags)
    if name.startswith('full') and FULL_CACHED:
        p = np.exp(P0['logp_hold__lgbm'].astype(np.float64))
        p /= p.sum(1, keepdims=True)
        src = '캐시'
    else:
        p = train_arm(cols)
        src = '학습'
    rep = axis_report(p, y_ho)
    results[name] = {'p': p, 'n_feat': len(cols), 'rep': rep}
    print(f"{name:<24} [{src}] {len(cols):>3}종  holdout ll {rep['총 ll']:.4f}  "
          f"(tie축 {rep['tie축']:.4f} / 승부축 {rep['승부축']:.4f})  {(time.time() - t0) / 60:.1f}분")
os.makedirs('outputs', exist_ok=True)
np.savez_compressed('outputs/ablation_preds.npz', y=y_ho,
                    **{f'p__{k}': v['p'] for k, v in results.items()})
print(f'\n총 {(time.time() - t_all) / 60:.1f}분 | saved: outputs/ablation_preds.npz')

In [ ]:
# ── 판정표 — Δ vs full + paired bootstrap CI + 축 분해 ──
full_key = 'full (전 유형 on)'
base_ll = per_row_ll(results[full_key]['p'], y_ho)
rows = []
for name, r in results.items():
    row = {'arm': name, 'n_feat': r['n_feat'], 'holdout ll': r['rep']['총 ll'],
           'tie축': r['rep']['tie축'], '승부축': r['rep']['승부축'],
           'tie AUC': r['rep']['tie AUC'], '승부 AUC': r['rep']['승부 AUC']}
    if name != full_key:
        m_, lo, hi = paired_bootstrap(per_row_ll(r['p'], y_ho), base_ll)
        row['Δ vs full'] = m_
        row['95% CI'] = f'[{lo:+.4f},{hi:+.4f}]'
        row['판정'] = (('유의하게 나쁨' if lo > 0 else '유의하게 좋음' if hi < 0 else '노이즈')
                       + (' ★' if (lo > 0 or hi < 0) and abs(m_) >= 0.0005 else ''))
    rows.append(row)
tbl = pd.DataFrame(rows).set_index('arm').sort_values('holdout ll')
tbl.to_csv('outputs/ablation_results.csv')
with pd.option_context('display.max_columns', None, 'display.width', 220):
    display(tbl.round(4))
print('saved: outputs/ablation_results.csv')
print('\n판정 규칙(§7): CI 가 0 을 포함하면 노이즈. ★ = 유의하고 |Δ| ≥ 문턱 0.0005.')

## 읽는 법

1. `− 유형` 의 Δ가 **+유의 ★** 면 그 유형이 실제로 벌고 있다 — 유지.
2. **노이즈**면 제거 후보 — 특히 `− 임베딩`(emb_model/ 불필요, GPU 불필요),
   `− TF-IDF 잠재`(svd.joblib 의존 축소)는 배포 단순화와 직결된다.
   의존성 전체(TF-IDF 코사인 포함)를 빼려면 FLAGS_CUSTOM 으로 조합 arm 을 추가할 것
   — 카탈로그의 의존성 컬럼이 기준이다.
3. `− 방향형` 성격의 판단: 방향(d_) 피쳐는 여러 유형에 분산돼 있으므로 축 분해 컬럼으로
   본다 — 승부축이 무너지는 arm 은 반대칭 재료를 뺀 것이다 (§0).
4. holdout 5,748 은 CI 가 넓다(SE ≈ 0.0058). 경계 판정(문턱 부근)은 stage2 급 표본이
   생길 때까지 보류하는 편이 안전하다.
5. **채택 변경**은 Train.ipynb 의 FLAGS 를 바꿔 **재학습 + meta.json 갱신**으로 반영한다 —
   이 노트북의 arm 모델은 저장하지 않는다(판정 전용).